# 03 — Exposure Labels v2

Identifies anchor posts and classifies users as exposed or unexposed using correct thread-level linking via `link_id`.

**Anchor post definition:**
- Post is in the anchor period: Sep 1–Nov 30, 2023 (cycle 1) or Sep 1–Nov 30, 2024 (cycle 2)
- Post matches negative keyword list (rejection language, re-applicant discourse, anxiety/stress/depression terms)
- Post `mean_mh_score > 0.45` from SVM classifiers (notebook 02)

**Exposed user**: commented on an anchor post thread (identified via `link_id`); anchor post authors excluded

**Unexposed user**: active in r/gradadmissions during Aug 1–May 31 of that cycle but never commented on an anchor thread

**Inputs:**
- `data/processed_v2/posts_clean.jsonl` + `comments_clean.jsonl` (from notebook 01)
- `models/clf_anxiety.joblib`, `clf_depression.joblib`, `clf_stress.joblib` (from notebook 02)

**Outputs:**
- `data/processed_v2/anchor_posts_v2.parquet`
- `data/processed_v2/exposure_labels_v2.parquet` — `author, exposed (bool), cycle`

In [1]:
import json
import pandas as pd
import numpy as np
import joblib
import re
from pathlib import Path

ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'data' / 'processed_v2'
MODEL_DIR = ROOT / 'models'

POSTS_PATH    = DATA_DIR / 'posts_clean.jsonl'
COMMENTS_PATH = DATA_DIR / 'comments_clean.jsonl'

# Cycle windows
CYCLES = {
    1: {
        'anchor_start': '2023-09-01',
        'anchor_end':   '2023-11-30',
        'active_start': '2023-08-01',
        'active_end':   '2024-05-31',
    },
    2: {
        'anchor_start': '2024-09-01',
        'anchor_end':   '2024-11-30',
        'active_start': '2024-08-01',
        'active_end':   '2025-05-31',
    },
}

MH_SCORE_THRESHOLD = 0.45


def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


print('Paths:')
print(' Posts:   ', POSTS_PATH)
print(' Comments:', COMMENTS_PATH)
print(' Models:  ', MODEL_DIR)
print(' Output:  ', DATA_DIR)

Paths:
 Posts:    /Users/sabelle/Projects/reddit-gradadmissions-distress/data/processed_v2/posts_clean.jsonl
 Comments: /Users/sabelle/Projects/reddit-gradadmissions-distress/data/processed_v2/comments_clean.jsonl
 Models:   /Users/sabelle/Projects/reddit-gradadmissions-distress/models
 Output:   /Users/sabelle/Projects/reddit-gradadmissions-distress/data/processed_v2


## 1) Load raw posts

In [2]:
posts = pd.DataFrame(load_jsonl(POSTS_PATH))
posts['created_dt'] = pd.to_datetime(posts['created_dt'], utc=True)
print(f'Clean posts loaded: {len(posts):,} from {posts["author"].nunique():,} unique authors')
print(f'Date range: {posts["created_dt"].min().date()} → {posts["created_dt"].max().date()}')

Clean posts loaded: 78,961 from 41,637 unique authors
Date range: 2023-08-01 → 2025-07-30


## 2) Filter to anchor periods and score with SVM classifiers

In [3]:
# Tag each post with its cycle (if in an anchor period)
def assign_cycle(dt):
    for cycle, w in CYCLES.items():
        if pd.Timestamp(w['anchor_start'], tz='UTC') <= dt <= pd.Timestamp(w['anchor_end'] + ' 23:59:59', tz='UTC'):
            return cycle
    return None

posts['cycle'] = posts['created_dt'].apply(assign_cycle)
anchor_candidates = posts[posts['cycle'].notna()].copy()

print(f'Posts in anchor periods: {len(anchor_candidates):,}')
print(anchor_candidates['cycle'].value_counts().sort_index())

Posts in anchor periods: 14,040
cycle
1.0    7009
2.0    7031
Name: count, dtype: int64


In [4]:
# Load SVM classifiers
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

texts = anchor_candidates['clean_text'].tolist()

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

anchor_candidates['anx_score'] = sigmoid(clf_anx.decision_function(texts))
anchor_candidates['dep_score'] = sigmoid(clf_dep.decision_function(texts))
anchor_candidates['str_score'] = sigmoid(clf_str.decision_function(texts))
anchor_candidates['mean_mh_score'] = anchor_candidates[['anx_score', 'dep_score', 'str_score']].mean(axis=1)

print(f'Scored {len(anchor_candidates):,} anchor-period posts')
print(anchor_candidates['mean_mh_score'].describe().round(4))

/Users/sabelle/venvs/jupyter/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


/Users/sabelle/venvs/jupyter/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


/Users/sabelle/venvs/jupyter/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/sabelle/venvs/jupyter/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Classifiers loaded.


Scored 14,040 anchor-period posts
count    14040.0000
mean         0.3557
std          0.0851
min          0.0871
25%          0.2974
50%          0.3492
75%          0.4072
max          0.7485
Name: mean_mh_score, dtype: float64


## 3) Apply keyword filter → anchor posts

In [5]:
NEGATIVE_KEYWORDS = [
    r'\breject(?:ed|ion)\b',
    r'\bdeclin(?:ed|ing)\b',
    r'\bwaitlist(?:ed)?\b',
    r'\bfunding\s+(?:lost|cut|removed|denied|gap|issue)\b',
    r'\bno\s+funding\b',
    r'\bstipend\b',
    r'\bwithdrew?\s+(?:offer|admission)\b',
    r'\bacceptance\s+rate\b',
    r'\bno\s+(?:offer|response|interview)\b',
    r'\bsilence\s+from\b',
    r'\bnot\s+(?:accepted|admitted|selected)\b',
    r'\bgave\s+up\b',
    r'\bmental\s+health\b',
    r'\banxi(?:ous|ety)\b',
    r'\bdepress(?:ed|ing|ion)\b',
    r'\bstress(?:ed|ful)?\b',
    r'\boverwhelm(?:ed|ing)\b',
    r'\bscared\b',
    r'\bworr(?:ied|ying)\b',
    r'\bfalling\s+apart\b',
    r'\bbreaking\s+down\b',
    r'\bcan(?:\'t|not)\s+(?:take|handle|cope)\b',
    r'\bno\s+chance\b',
    r'\bnot\s+good\s+enough\b',
    r'\bgave\s+up\b',
    r'\bregret\b',
    r'\bfailed\b',
    r'\bimposter\b',
]

MH_THRESHOLD = 0.5   # per-dimension threshold (paper: "above threshold on at least one dimension")

keyword_pattern = re.compile('|'.join(NEGATIVE_KEYWORDS), re.IGNORECASE)

anchor_candidates['has_neg_keyword'] = anchor_candidates['clean_text'].str.contains(
    keyword_pattern, na=False
)

# Paper §4.3: retain posts scoring above threshold on AT LEAST ONE dimension (OR logic)
anchor_posts = anchor_candidates[
    anchor_candidates['has_neg_keyword'] & (
        (anchor_candidates['anx_score'] > MH_THRESHOLD) |
        (anchor_candidates['dep_score'] > MH_THRESHOLD) |
        (anchor_candidates['str_score'] > MH_THRESHOLD)
    )
].copy()

print(f'Anchor posts identified: {len(anchor_posts):,}')
print(anchor_posts['cycle'].value_counts().sort_index())
print(f'Unique anchor authors: {anchor_posts["author"].nunique():,}')
print(f'Mean mh_score of anchor posts: {anchor_posts["mean_mh_score"].mean():.4f}')
print(f'\nScore breakdown (fraction above {MH_THRESHOLD} per dimension):')
for dim in ['anx_score', 'dep_score', 'str_score']:
    n = (anchor_posts[dim] > MH_THRESHOLD).sum()
    print(f'  {dim}: {n:,} ({100*n/len(anchor_posts):.1f}%)')

Anchor posts identified: 597
cycle
1.0    261
2.0    336
Name: count, dtype: int64
Unique anchor authors: 548
Mean mh_score of anchor posts: 0.5248

Score breakdown (fraction above 0.5 per dimension):
  anx_score: 391 (65.5%)
  dep_score: 253 (42.4%)
  str_score: 511 (85.6%)


In [6]:
# Save anchor posts
anchor_posts[[
    'id', 'author', 'created_dt', 'cycle', 'clean_text',
    'anx_score', 'dep_score', 'str_score', 'mean_mh_score', 'score', 'num_comments'
]].to_parquet(DATA_DIR / 'anchor_posts_v2.parquet', index=False)
print('Saved anchor_posts_v2.parquet')

# Anchor post ID sets per cycle
anchor_ids_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['id'])
    for cycle in [1, 2]
}
print(f'Anchor IDs — cycle 1: {len(anchor_ids_by_cycle[1]):,}, cycle 2: {len(anchor_ids_by_cycle[2]):,}')

# Anchor authors per cycle (to exclude from exposed set)
anchor_authors_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['author'])
    for cycle in [1, 2]
}

Saved anchor_posts_v2.parquet
Anchor IDs — cycle 1: 261, cycle 2: 336


## 4) Load comments → identify exposed users via `link_id`

In [7]:
comments = pd.DataFrame(load_jsonl(COMMENTS_PATH))
comments['created_dt'] = pd.to_datetime(comments['created_dt'], utc=True)
# post_id is already derived from link_id by notebook 01
print(f'Clean comments loaded: {len(comments):,} from {comments["author"].nunique():,} unique authors')

Clean comments loaded: 467,986 from 79,573 unique authors


In [8]:
# All anchor post IDs across both cycles
all_anchor_ids = anchor_ids_by_cycle[1] | anchor_ids_by_cycle[2]

# Comments on anchor posts
anchor_comments = comments[comments['post_id'].isin(all_anchor_ids)].copy()
print(f'Comments on anchor posts: {len(anchor_comments):,}')

# Tag which cycle each anchor comment belongs to
def comment_cycle(post_id):
    if post_id in anchor_ids_by_cycle[1]: return 1
    if post_id in anchor_ids_by_cycle[2]: return 2
    return None

anchor_comments['cycle'] = anchor_comments['post_id'].apply(comment_cycle)
print(anchor_comments['cycle'].value_counts().sort_index())

Comments on anchor posts: 4,639
cycle
1    1785
2    2854
Name: count, dtype: int64


In [9]:
# Exposed users per cycle: commenters on anchor posts
exposed_records = []

for cycle in [1, 2]:
    cycle_comments = anchor_comments[anchor_comments['cycle'] == cycle]
    excluded = anchor_authors_by_cycle[cycle]
    exposed_authors = set(cycle_comments['author']) - excluded
    for author in exposed_authors:
        exposed_records.append({'author': author, 'exposed': True, 'cycle': cycle, 'exposure_prob': 1.0})
    print(f'Cycle {cycle} — exposed users: {len(exposed_authors):,} '
          f'(excluded {len(set(cycle_comments["author"]) & excluded):,} anchor authors)')

exposed_df = pd.DataFrame(exposed_records)
print(f'\nTotal exposed: {len(exposed_df):,}')

Cycle 1 — exposed users: 835 (excluded 105 anchor authors)
Cycle 2 — exposed users: 1,198 (excluded 162 anchor authors)

Total exposed: 2,033


## 5) Identify unexposed users — same-week active, no anchor thread engagement

# Paper §4.3: unexposed = active in r/gradadmissions during the SAME WEEK as an anchor event
# (posted at least one comment or post that week) and did not engage with any anchor thread

In [10]:
import numpy as np

# iso_week = "YYYY-WNN" string
def iso_week(dt):
    return dt.strftime('%G-W%V')

posts['iso_week']    = posts['created_dt'].apply(iso_week)
comments['iso_week'] = comments['created_dt'].apply(iso_week)

anchor_posts['iso_week'] = anchor_posts['created_dt'].apply(iso_week)
anchor_weeks_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['iso_week'])
    for cycle in [1, 2]
}

# Users active (posted or commented) in any anchor week, per cycle
unexposed_records = []

# We calculate lambda from the empirical half-life of 8.27 hours
decay_lambda = np.log(2) / 8.27
print(f"Using exponential decay factor lambda: {decay_lambda:.4f}")

for cycle in [1, 2]:
    anchor_weeks = anchor_weeks_by_cycle[cycle]
    exposed_this_cycle = set(exposed_df[exposed_df['cycle'] == cycle]['author'])
    
    # Filter anchors for this cycle
    cycle_anchors = anchor_posts[anchor_posts['cycle'] == cycle]

    # Posts and comments during anchor weeks
    cycle_posts = posts[posts['iso_week'].isin(anchor_weeks)]
    cycle_comments = comments[comments['iso_week'].isin(anchor_weeks)]
    
    active_this_week = set(cycle_posts['author']) | set(cycle_comments['author'])
    unexposed_authors = active_this_week - exposed_this_cycle

    # Create a mapping for quick lookup: author -> list of activity timestamps
    author_activities = {}
    for _, row in cycle_posts.iterrows():
        author_activities.setdefault(row['author'], []).append(row['created_dt'])
    for _, row in cycle_comments.iterrows():
        author_activities.setdefault(row['author'], []).append(row['created_dt'])

    anchor_dts = cycle_anchors['created_dt'].tolist()

    for author in unexposed_authors:
        acts = author_activities.get(author, [])
        if not acts or not anchor_dts:
            unexposed_records.append({'author': author, 'exposed': False, 'cycle': cycle, 'exposure_prob': 0.0})
            continue
            
        min_delta_h = float('inf')
        for act in acts:
            for anc in anchor_dts:
                dh = abs((act - anc).total_seconds()) / 3600.0
                if dh < min_delta_h:
                    min_delta_h = dh
                    
        prob = np.exp(-decay_lambda * min_delta_h)
        unexposed_records.append({'author': author, 'exposed': False, 'cycle': cycle, 'exposure_prob': prob})

    print(f'Cycle {cycle} — exposed: {len(exposed_this_cycle):,} | unexposed: {len(unexposed_authors):,}')

unexposed_df = pd.DataFrame(unexposed_records)
print(f'\nTotal unexposed: {len(unexposed_df):,}')
print(unexposed_df['exposure_prob'].describe())

Using exponential decay factor lambda: 0.0838


Cycle 1 — exposed: 835 | unexposed: 9,072


Cycle 2 — exposed: 1,198 | unexposed: 10,625

Total unexposed: 19,697
count    19697.000000
mean         0.771418
std          0.250444
min          0.000255
25%          0.663191
50%          0.868939
75%          0.959763
max          1.000000
Name: exposure_prob, dtype: float64


## 6) Combine and save exposure labels

In [11]:
exposure_df = pd.concat([exposed_df, unexposed_df], ignore_index=True)

# A user could appear in both cycles — that's fine, keep both rows
print(f'Total exposure records: {len(exposure_df):,}')
print(f'Unique users: {exposure_df["author"].nunique():,}')
print('\nExposed vs unexposed by cycle:')
print(exposure_df.groupby(['cycle', 'exposed']).size().unstack(fill_value=0))

exposure_df.to_parquet(DATA_DIR / 'exposure_labels_v2.parquet', index=False)
print('\nSaved exposure_labels_v2.parquet')

Total exposure records: 21,730
Unique users: 20,932

Exposed vs unexposed by cycle:
exposed  False  True 
cycle                
1         9072    835
2        10625   1198

Saved exposure_labels_v2.parquet


## 7) Quick sanity checks

In [12]:
# Quick sanity checks
print(f'Average exposure prob among all active users: {exposure_df["exposure_prob"].mean():.4f}')
print(exposure_df.groupby(['cycle', 'exposed'])['exposure_prob'].describe().round(4))

Average exposure prob among all active users: 0.7928
                 count    mean     std     min     25%     50%     75%  max
cycle exposed                                                              
1     False     9072.0  0.7252  0.2879  0.0003  0.5810  0.8355  0.9504  1.0
      True       835.0  1.0000  0.0000  1.0000  1.0000  1.0000  1.0000  1.0
2     False    10625.0  0.8109  0.2052  0.0403  0.7232  0.8898  0.9650  1.0
      True      1198.0  1.0000  0.0000  1.0000  1.0000  1.0000  1.0000  1.0
